## CatBoostClassifier

Данные полученных экспериментов находятся в папке `results`.

In [1]:
# %pip install catboost

import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, auc, balanced_accuracy_score, accuracy_score

In [2]:
df_2020 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2020_preprocessed.csv')
df_merged = pd.read_csv(r'..\..\preprocessing\data\processed\df_merged.csv')

df_2020.head()

,HeartDisease,BMI,SmokerStatus,AlcoholDrinkers,HadStroke,PhysicalHealthDays,MentalHealthDays,DifficultyWalking,Sex,AgeCategory,RaceEthnicityCategory,HadDiabetes,PhysicalActivities,GeneralHealth,SleepHours,HadAsthma,HadKidneyDisease,HadSkinCancer
0,0,16.60,1,0,0,3.0,30.0,0,Female,55-59,White,Yes,1,Very good,5.0,1,0,1
1,0,20.34,0,0,1,0.0,0.0,0,Female,80 or older,White,No,1,Very good,7.0,0,0,0
2,0,26.58,1,0,0,20.0,30.0,0,Male,65-69,White,Yes,1,Fair,8.0,1,0,0
3,0,24.21,0,0,0,0.0,0.0,0,Female,75-79,White,No,0,Good,6.0,0,0,1
4,0,23.71,0,0,0,28.0,0.0,1,Female,40-44,White,No,1,Very good,8.0,0,0,0


In [3]:
results = pd.DataFrame()

### Обучение на наборе 2020 года

In [4]:
df_2020['AgeCategory'] = df_2020['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_2020['GeneralHealth'] = df_2020['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

In [5]:
SEED = 67

X = df_2020.drop('HeartDisease', axis=1)
y = df_2020['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [6]:
model = CatBoostClassifier(
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    random_seed=SEED,
    verbose=0,
    use_best_model=True
)

model.fit(
    X_train,
    y_train,
    cat_features=['Sex', 'RaceEthnicityCategory', 'HadDiabetes'],
    verbose=100,
    eval_set=(X_valid, y_valid)
)

Learning rate set to 0.124346
0:	learn: 0.7994526	test: 0.7969145	best: 0.7969145 (0)	total: 377ms	remaining: 6m 16s
100:	learn: 0.8309849	test: 0.8242785	best: 0.8242785 (100)	total: 20.5s	remaining: 3m 2s
200:	learn: 0.8368839	test: 0.8237190	best: 0.8243028 (121)	total: 42.6s	remaining: 2m 49s
300:	learn: 0.8410960	test: 0.8228361	best: 0.8243028 (121)	total: 1m 5s	remaining: 2m 32s
400:	learn: 0.8450523	test: 0.8217686	best: 0.8243028 (121)	total: 1m 28s	remaining: 2m 12s
500:	learn: 0.8491015	test: 0.8209914	best: 0.8243028 (121)	total: 1m 51s	remaining: 1m 50s
600:	learn: 0.8525523	test: 0.8197095	best: 0.8243028 (121)	total: 2m 14s	remaining: 1m 28s
700:	learn: 0.8558338	test: 0.8186490	best: 0.8243028 (121)	total: 2m 37s	remaining: 1m 7s
800:	learn: 0.8586619	test: 0.8178323	best: 0.8243028 (121)	total: 3m 2s	remaining: 45.3s
900:	learn: 0.8617331	test: 0.8170755	best: 0.8243028 (121)	total: 3m 25s	remaining: 22.6s
999:	learn: 0.8645092	test: 0.8163338	best: 0.8243028 (121)	tot

CatBoostClassifier(auto_class_weights='Balanced', eval_metric='PRAUC', random_seed=67, use_best_model=True, verbose=0)

In [7]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

In [8]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

metrics2020 = {
    'data': '2020',
    'model': 'CatBoostClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba),
    'f1_score': f1_score(y_test, y_pred),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred)
}

In [9]:
metrics2020

{'data': '2020',
 'model': 'CatBoostClassifier',
 'roc_auc_score': 0.8433346727927207,
 'f1_score': 0.34188846641318127,
 'pr_auc_score': 0.3510545302568115,
 'balanced_accuracy_score': 0.7664086849439846}

In [10]:
fi2020 = pd.DataFrame({
    'data': '2020',
    'feature': X.columns,
    'importances': model.get_feature_importance()
}).sort_values('importances', ascending=False)

### Обучение на общем наборе 2020+2022

In [11]:
df_merged['AgeCategory'] = df_merged['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_merged['GeneralHealth'] = df_merged['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

df_merged = df_merged.fillna('nan')

In [12]:
SEED = 67

X = df_merged.drop(['HeartDisease'], axis=1)
y = df_merged['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [13]:
model = CatBoostClassifier(
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    random_seed=SEED,
    verbose=0,
    use_best_model=True
)

model.fit(
    X_train,
    y_train,
    cat_features=['Sex', 'RaceEthnicityCategory', 'HadDiabetes'],
    verbose=100,
    eval_set=(X_valid, y_valid)
)

Learning rate set to 0.151162
0:	learn: 0.7993994	test: 0.7986276	best: 0.7986276 (0)	total: 379ms	remaining: 6m 19s
100:	learn: 0.8269847	test: 0.8230076	best: 0.8230076 (100)	total: 41s	remaining: 6m 5s
200:	learn: 0.8302671	test: 0.8230097	best: 0.8231171 (164)	total: 1m 24s	remaining: 5m 33s
300:	learn: 0.8331183	test: 0.8226708	best: 0.8231171 (164)	total: 2m 7s	remaining: 4m 56s
400:	learn: 0.8353875	test: 0.8222384	best: 0.8231171 (164)	total: 2m 51s	remaining: 4m 15s
500:	learn: 0.8375663	test: 0.8217327	best: 0.8231171 (164)	total: 3m 35s	remaining: 3m 34s
600:	learn: 0.8395622	test: 0.8214486	best: 0.8231171 (164)	total: 4m 19s	remaining: 2m 52s
700:	learn: 0.8415793	test: 0.8208506	best: 0.8231171 (164)	total: 5m 4s	remaining: 2m 10s
800:	learn: 0.8436708	test: 0.8204281	best: 0.8231171 (164)	total: 5m 47s	remaining: 1m 26s
900:	learn: 0.8453958	test: 0.8201609	best: 0.8231171 (164)	total: 6m 28s	remaining: 42.7s
999:	learn: 0.8470939	test: 0.8197120	best: 0.8231171 (164)	to

CatBoostClassifier(auto_class_weights='Balanced', eval_metric='PRAUC', random_seed=67, use_best_model=True, verbose=0)

In [14]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

metrics_merged = {
    'data': 'merged',
    'model': 'CatBoostClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba),
    'f1_score': f1_score(y_test, y_pred),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred)
}

In [15]:
metrics_merged

{'data': 'merged',
 'model': 'CatBoostClassifier',
 'roc_auc_score': 0.8373789861181673,
 'f1_score': 0.34432699083861873,
 'pr_auc_score': 0.3419250040461593,
 'balanced_accuracy_score': 0.7595204857526182}

In [16]:
fi_merged = pd.DataFrame({
    'data': 'merged',
    'feature': X.columns,
    'importances': model.get_feature_importance()
}).sort_values('importances', ascending=False)

### Обучение на наборе 2022 года

In [17]:
df_2022 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2022_preprocessed.csv')

df_2022['AgeCategory'] = df_2022['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_2022['GeneralHealth'] = df_2022['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

df_2022['RemovedTeeth'] = df_2022['RemovedTeeth'].map({
    'None of them': 0,
    '1 to 5': 1,
    '6 or more, but not all': 2,
    'All': 3
})

df_2022['LastCheckupTime'] = df_2022['LastCheckupTime'].map({
    'Within past year (anytime less than 12 months ago)': 0,
    'Within past 2 years (1 year but less than 2 years ago)': 1,
    'Within past 5 years (2 years but less than 5 years ago)': 2,
    '5 or more years ago': 3,
})

df_2022['CovidPos'] = df_2022['CovidPos'].map({
    'No': 0,
    'Yes': 1,
})

df_2022 = df_2022.fillna('nan')

In [18]:
SEED = 67

X = df_2022.drop(['HeartDisease', 'HadAngina', 'HadHeartAttack'], axis=1)
y = df_2022['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

model2022 = CatBoostClassifier(
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    random_seed=SEED,
    verbose=0,
    use_best_model=True
)

model2022.fit(
    X_train,
    y_train,
    cat_features=['State',
                  'Sex',
                  'HadDiabetes',
                  'ECigaretteUsage',
                  'RaceEthnicityCategory',
                  'TetanusLast10Tdap'],
    verbose=100,
    eval_set=(X_valid, y_valid)
)

Learning rate set to 0.130201
0:	learn: 0.7988863	test: 0.8021277	best: 0.8021277 (0)	total: 465ms	remaining: 7m 44s
100:	learn: 0.8361014	test: 0.8365905	best: 0.8365905 (100)	total: 37.4s	remaining: 5m 32s
200:	learn: 0.8425809	test: 0.8370811	best: 0.8371572 (176)	total: 1m 13s	remaining: 4m 50s
300:	learn: 0.8476223	test: 0.8370226	best: 0.8372290 (224)	total: 1m 49s	remaining: 4m 14s
400:	learn: 0.8517500	test: 0.8364626	best: 0.8372290 (224)	total: 2m 28s	remaining: 3m 42s
500:	learn: 0.8559317	test: 0.8359892	best: 0.8372290 (224)	total: 3m 7s	remaining: 3m 7s
600:	learn: 0.8598741	test: 0.8352728	best: 0.8372290 (224)	total: 3m 48s	remaining: 2m 31s
700:	learn: 0.8637190	test: 0.8344713	best: 0.8372290 (224)	total: 4m 28s	remaining: 1m 54s
800:	learn: 0.8673510	test: 0.8338261	best: 0.8372290 (224)	total: 5m 8s	remaining: 1m 16s
900:	learn: 0.8707847	test: 0.8333848	best: 0.8372290 (224)	total: 5m 48s	remaining: 38.3s
999:	learn: 0.8740772	test: 0.8328699	best: 0.8372290 (224)	

CatBoostClassifier(auto_class_weights='Balanced', eval_metric='PRAUC', random_seed=67, use_best_model=True, verbose=0)

In [19]:
y_proba = model2022.predict_proba(X_test)[:, 1]
y_pred = model2022.predict(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

metrics_2022 = {
    'data': '2022',
    'model': 'CatBoostClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba),
    'f1_score': f1_score(y_test, y_pred),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred)
}

In [20]:
fi2022 = pd.DataFrame({
    'data': '2022',
    'feature': X.columns,
    'importances': model2022.get_feature_importance()
}).sort_values('importances', ascending=False)

In [21]:
results = pd.DataFrame([
    metrics2020,
    metrics_2022,
    metrics_merged,
])

In [22]:
results.to_csv(r'../results/catboost.csv')

In [24]:
fi2020.to_csv('../results/feature_importance_2020.csv', index=False)
fi_merged.to_csv('../results/feature_importance_merged.csv', index=False)
fi2022.to_csv('../results/feature_importance_2022.csv', index=False)